# Wasserstein-Fisher-Rao Gradient Flow with Adaptive Biasing Force

## Setup

### Potential

$$
V(x, y) = 3 e^{-x^2} \left( e^{- \left(y - \frac{1}{3}\right)^2} - e^{- \left(y - \frac{5}{3} \right)^2} \right) - 5 e^{-y^2} \left( e^{-(x - 1)^2} + e^{- (x + 1)^2} \right) + 0.2x^4 + 0.2 \left(y - \frac{1}{3}\right)^4.
$$

### Boltzmann distribution

$$
\pi_\beta \propto \exp (-\beta V(x, y)).
$$

![Target density with $\beta = 4$](fig/target_density_beta4.png)

### Reaction coordinate and free energy

The reaction coordinate is on the $x$-axis:
$$
\xi(x, y) = x,
$$
and the free energy is hence defined as
$$
F(z) = -\frac{1}{\beta} \ln{\frac{Z(z)}{Z}}, \qquad Z(z) = \int_{-5}^5 \exp(-\beta V(x, y)) dy, \qquad Z = \int_{-5}^5 \int_{-5}^5 \exp (-\beta V(x, y)) dx dy.
$$

We then compute that $\nabla\xi = (1, 0)$ and $|\nabla\xi| = 1$, hence the local mean force is simply 
$$f (x, y) = \partial_x V(x, y),$$
and furthermore
$$
F'(z) = \mathbb{E}_\pi [\partial_x V(X, Y) \mid X = z] = \frac{\int_{-5}^5 \partial_x V(z, y) \exp(-\beta V(z, y)) dy}{\int_{-5}^5 \exp(-\beta V(z, y)) dy}.
$$

## Current Plan

We restrict the domain in $[-5, 5]^2$ using the similar strategy as in `eABF.ipynb`, and let $\rho_t(x, y)$ be the time-dependent density on domain $[-5, 5]^2$. The marginal distribution on $x$-coordinate is hence
$$
p_t (x) = \int_{-5}^5 \rho_t (x, y) dy.
$$

### Langevin dynamics with ABF
Let $A_t$ be the ABF estimate of the free energy, with its derivative $A_t'(x)$ estimating $F'(x)$, then the Langevin dynamics with ABF reads
$$
\begin{cases}
d X_t = (-\partial_x V(X_t, Y_t) + A_t' (X_t)) dt + \sqrt{\frac{2}{\beta}} d W_t^x, \\
d Y_t = -\partial_y V(X_t, Y_t) dt + \sqrt{\frac{2}{\beta}} d W_t^y,
\end{cases}
$$
with the (ideal) ABF estimator given by
$$
A_t' (x) = \mathbb{E}_{\rho_t} [\partial_x V(X, Y) \mid X = x] = \frac{\int_{-5}^5 \partial_x V(x, y) \rho_t (x, y) dy}{\int_{-5}^5 \rho_t (x, y) dy},
$$
and the kernel estimation is 
$$
A_t' (x) = \frac{\int\int K_h (x - x') \partial_{x'} V(x', y') \rho_t (x', y') dx'dy'}{\int\int K_h (x - x') \rho_t (x', y') dx'dy'}.
$$

### Fisher-Rao (Birth/Death)
If the ABF is exact, i.e., $A(x) = F(x)$, then the biased density would be $\pi_A (x, y) \propto \exp (-\beta (V(x, y) - F(x)))$, whose marginal in $x$-coordinate is 
$$
\int_{-5}^5 \exp (-\beta (V(x, y) - F(x))) dy = \exp (\beta F(x)) Z(x) = Z,
$$
so the marginal along $x$ is hence the (flattened) uniform distribution $u(x) = \frac{1}{10} \mathbf{1}_{[-5, 5]} (x)$, the birth-death term reads
$$
\mathcal{R}[\rho_t](x, y) = -\gamma \rho_t (x, y) \left( \ln{\frac{p_t(x)}{u(x)}} - \int_{-5}^5 p_t(z) \ln{\frac{p_t(z)}{u(z)}} dz \right)
$$
with $\int\int \mathcal{R}[\rho_t](x, y)dxdy = 0$.

### PDE
$$
\partial_t \rho_t = \frac{1}{\beta} (\partial_{xx} \rho_t + \partial_{yy} \rho_t) + \partial_x ((\partial_x V - A_t'(x)) \rho_t) + \partial_y ((\partial_y V) \rho_t) - \gamma \rho_t (x, y) \left( \ln{\frac{p_t(x)}{u(x)}} - \int_{-5}^5 p_t(z) \ln{\frac{p_t(z)}{u(z)}} dz \right),
$$
or the kernelized version
$$
\begin{align*}
\partial_t \rho_t &= \frac{1}{\beta} (\partial_{xx} \rho_t + \partial_{yy} \rho_t) + \partial_x ((\partial_x V - A_t'(x)) \rho_t) + \partial_y ((\partial_y V) \rho_t)\\
&\qquad - \gamma \rho_t (x, y) \left( \ln{\frac{(K_\eta * p_t)(x)}{u(x)}} + K_\eta * \left( \frac{p_t}{K_\eta * p_t} \right)(x) - \int_{-5}^5 p_t(z) \ln{\frac{(K_\eta * p_t)(z)}{u(z)}} dz - 1 \right).
\end{align*}
$$

## Numerical implementation with $N$ particles

Setup: $N$ particles $(X_n^i, Y_n^i)$ at timestep $n$

### 1. Estimating ABF
$$
\hat A_n'(x) = \frac{\sum_{i=1}^N K_h (x - X_n^i) \partial_x V(X_n^i, Y_n^i)}{\sum_{i=1}^N K_h (x - X_n^i)}.
$$

### 2. Langevin dynamics with ABF
$$
\begin{cases}
\tilde X_{n+1}^i = X_n^i + (-\partial_x V(X_n^i, Y_n^i) + \hat A_t'(X_n^i)) \Delta t + \sqrt{2\Delta t/\beta} \xi_n^{i, x},\\
\tilde Y_{n+1}^i = Y_n^i - \partial_y V(X_n^i, Y_n^i) \Delta t + \sqrt{2\Delta t/\beta} \xi_n^{i, y}.
\end{cases}
$$

### 3. Estimate marginal on $x$-coordinate
$$
\hat p_{n+1}(x) = \frac{1}{N} \sum_{i=1}^N K_\eta (x - \tilde X_{n+1}^i).
$$

### 4. Birth-death on $x$-coordinate
For particle $i$, define its birth-death score
$$
\hat S_{n+1} (\tilde X_{n+1}^i) = \ln{\hat p_{n+1} (\tilde X_{n+1}^i)} - \frac{1}{N} \sum_{j=1}^N \ln{\hat p_{n+1} (X_{n+1}^j)}
$$
or the kernelized version
$$
\hat S_{n+1}^\eta (\tilde X_n^i) = \ln{\hat p_{n+1} (\tilde X_{n+1}^i)} + \frac{1}{N} \sum_{j=1}^N \frac{K_\eta (\tilde X_{n+1}^i - \tilde X_{n+1}^j)}{\hat p_{n+1} (X_{n+1}^j)} - \frac{1}{N} \sum_{j=1}^N \ln{\hat p_{n+1} (X_{n+1}^j)} - 1,
$$
then:
* if $\hat S_{n+1} (\tilde X_{n+1}^i) > 0$, kill particle $i$ with probability $1 - e^{-\gamma \hat S_{n+1} (\tilde X_{n+1}^i) \Delta t}$ and clone another particle with uniform probability;
* if $\hat S_{n+1} (\tilde X_{n+1}^i) < 0$, clone particle $i$ with probability $1 - e^{\gamma \hat S_{n+1} (\tilde X_{n+1}^i) \Delta t}$ and kill another particle with uniform probability.

After birth-death step, we denote the resulting particle system by
$$(X_{n+1}^i, Y_{n+1}^i), \qquad i = 1, \ldots, N.$$